In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os 
import sys 
sys.path.append('..')

import matplotlib.pyplot as plt 
import numpy as np
import torch

import hippocampalseq as hse
import hippocampalseq.utils as hseu
import hippocampalseq.preprocessing as hsep
import hippocampalseq.models as hsem
import hippocampalseq.plotting as hsepl

In [ ]:
theta_time_window_s  = 20/1000#60 / 1000
theta_time_window_advance_s  = 5/1000#60 / 1000

ripple_time_window_s  = 3.0 / 1000


bin_size = 2
data_path = os.path.realpath("../data")
rat_name = "Harpy"
session = 1
track_type = "Linear"

if track_type == "Linear":
    environment_size = None
else:
    environment_size =  hsep.EnvironmentSize((0,200), (0,200))

nplot = 20

In [ ]:
(
    raw_data,
    place_field_data,
) = hse.load_raw_data(
    data_path,
    rat_name,
    session,
    track_type=track_type,
    environment_size=environment_size,
    bin_size_cm=bin_size,
    placefield_kwargs = {
        'place_field_posterior' : False,
        'velocity_cutoff'       : 10.0,
        "flatten_linear"        : True,
    },

)

theta_data = hse.process_theta(
    raw_data,
    place_field_data, 
    velocity_cutoff=10.0,
    theta_kwargs = {
        'time_window_s'         : theta_time_window_s,
        'time_window_advance_s' : theta_time_window_advance_s 
    }
)

%store raw_data place_field_data theta_data 

In [ ]:
%store -r raw_data place_field_data theta_data
place_fields = place_field_data.place_fields[place_field_data.place_cell_ids]

In [ ]:
bmap = hsem.BayesianMAP(
    place_fields=place_fields,
    dt=theta_time_window_s,
    bin_size=bin_size
)
map_decoded = bmap.fit(theta_data.spikes)

In [ ]:
def bin_posterior_phase_gaussian(
    decoded_trajectory: list[np.ndarray],
    decoded_cumulative_probability: list[np.ndarray],
    spikemats: list[np.ndarray],
    environment_size: hsep.EnvironmentSize,
    segments: list[nap.TsdFrame],
    *,
    spike_count_percentile: float = 66.67,
    bin_size_cm: int = 2,
    phase_bin_width: int = 10,
    distance_edges=None,
):
    phase_col = 'Phase Deg'
    head_direction_col = 'Head direction'

    oscillation_spikes = []
    for gt in theta_data.ground_truth:
        oscs = set(gt['Oscillation Number'])
        oscillation_spikes.append(
        )


In [ ]:
momentum = hsem.Momentum(
    dt=theta_time_window_s,
    bin_size=bin_size,
    environment_size=raw_data.environment_size,
    place_fields=place_fields,
    spikemat_train=theta_data.spikes
)
momentum_results= momentum.fit()

In [ ]:
import numpy as np
import pynapple as nap
from scipy.stats import multivariate_normal

def bin_posterior_phase_gaussian(
    posterior_means: list[np.ndarray],
    posterior_covariances: list[np.ndarray]|None,
    environment_size: hsep.EnvironmentSize,
    segments: list[nap.TsdFrame],
    *,
    phase_col: str = "Phase Deg",
    head_direction_col: str ="Head direction",
    bin_size_cm: int = 2,
    phase_bin_width: int = 10,
    distance_edges=None,
):
    """
    Evaluate posterior distributions across an environment, compute signed
    point-to-subject distances, and aggregate posterior-weighted distances by
    oscillation phase.

    A point is assigned a positive distance if it is in front of the subject's
    head direction, and a negative distance if it is behind. "In front" is
    determined by the dot product between the point displacement vector and
    the unit head-direction vector.

    Args: 
        posterior_means (list[np.ndarray]): One posterior mean per time step. 
        posterior_covariances (list[np.ndarray]|None): One posterior covariance per time step. If None, set to 0 covariance.
        environment_size (hsep.EnvironmentSize): Size of the environment.
        true_dfs (list[nap.TsdFrame]): True position, head direction, and theta phase columns must be included.
        phase_col (str): Column containing wave phase in degrees. Defaults to 'Phase Deg'.
        head_direction_col (str): Column containing head direction in degrees, where 0° points along +x
            and angles increase counter-clockwise. Defaults to 'Head direction'.
        bin_size_cm (float): Size of each spatial bin in centimeters. Defaults to 2.
        phase_bin_width (float): Width of phase bins in degrees. Defaults to 10.

    Returns:

    """
    if posterior_covariances is None:
        # Good for point estimates
        posterior_covariances = [
            np.zeros((len(m), m.shape[1], m.shape[1]))
            for m in posterior_means
        ]
    if not (
        len(posterior_means)
        == len(posterior_covariances)
        == len(segments)
    ):
        raise ValueError(
            "posterior_means, posterior_covariances, and segments "
            "must have the same number of elements."
        )

    position_cols = environment_size.axes()

    grid_points = hseu.make_ndgrid(environment_size, bin_size_cm).numpy()
    spatial_dim = len(environment_size)
    if len(environment_size) == 1:
        axis = environment_size.x if environment_size.x else environment_size.y
        max_distance = axis[1] - axis[0]
        cell_volume = max_distance / (len(grid_points) - 1)
    else:
        x = environment_size.x
        y = environment_size.y
        max_distance = np.hypot(
            x[1] - x[0],
            y[1] - y[0]
        )
        cell_volume = (
            (x[1] - x[0]) / (len(np.unique(grid_points[:, 0])) - 1)
            * (y[1] - y[0]) / (len(np.unique(grid_points[:, 1])) - 1)
        )

    if distance_edges is None:
        distance_edges = np.linspace(-max_distance, max_distance, 101)
    else:
        distance_edges = np.asarray(distance_edges, dtype=float)

    if np.any(np.diff(distance_edges) <= 0):
        raise ValueError("distance_edges must be strictly increasing.")

    phase_edges = np.arange(0.0, 360.0 + phase_bin_width, phase_bin_width)
    n_distance_bins = len(distance_edges) - 1
    n_phase_bins = len(phase_edges) - 1

    posterior_mass = np.zeros((n_phase_bins, n_distance_bins), dtype=float)
    phase_counts = np.zeros(n_phase_bins, dtype=int)

    for means_segment, covs_segment, frame in zip(
        posterior_means,
        posterior_covariances,
        segments,
    ):
        if means_segment.ndim == 3:
            means_segment = np.squeeze(means_segment, axis=-1)
        means_segment = hseu.atleast_2d(means_segment)

        if means_segment.shape[1] != spatial_dim:
            raise ValueError(
                f"Means have dimension {means_segment.shape[1]}, but "
                f"environment_size describes a {spatial_dim}D space."
            )

        phase = np.mod(frame[phase_col].values, 360.0)
        head_direction = np.deg2rad(frame[head_direction_col].values)

        true_position = hseu.atleast_2d(frame[position_cols])
        
        n_time = len(means_segment)
        if not (
            len(phase) == len(head_direction) == len(true_position) == n_time
            and covs_segment.shape[0] == n_time
        ):
            raise ValueError(
                "Each segment's posterior arrays and TsdFrame must have "
                "the same number of time points."
            )

        phase_bin = np.clip(
            np.digitize(phase, phase_edges, right=False) - 1,
            0,
            n_phase_bins - 1,
        )

        for t in range(n_time):
            displacement = grid_points - true_position[t]

            if displacement.shape[1] == 1:
                unsigned_distance = np.abs(displacement[:, 0])

                # Direction is +1 for directions facing roughly +x and
                # -1 for directions facing roughly -x.
                direction = 1.0 if np.cos(head_direction[t]) >= 0 else -1.0
                signed_distance = unsigned_distance * np.sign(
                    displacement[:, 0] * direction
                )
            else:
                unsigned_distance = np.linalg.norm(displacement, axis=1)

                # 0 degrees points along +x; 90 degrees points along +y.
                heading_vector = np.array(
                    [np.cos(head_direction[t]), np.sin(head_direction[t])]
                )

                # Positive = grid point in front of the head direction.
                signed_distance = unsigned_distance * np.sign(
                    displacement @ heading_vector
                )

            density = multivariate_normal(
                mean=means_segment[t],
                cov=covs_segment[t]
            ).pdf(grid_points)

            # Discrete approximation of posterior probability mass over
            # each environment grid cell; renormalizing avoids edge loss
            # when posterior density extends beyond the environment.
            point_mass = density * cell_volume
            point_mass /= point_mass.sum()

            distance_bin = np.digitize(
                signed_distance,
                distance_edges,
                right=False,
            ) - 1

            valid = (distance_bin >= 0) & (distance_bin < n_distance_bins)
            np.add.at(
                posterior_mass[phase_bin[t]],
                distance_bin[valid],
                point_mass[valid],
            )
            phase_counts[phase_bin[t]] += 1

    return {
        "posterior_mass": posterior_mass,
        "phase_edges": phase_edges,
        "phase_centers": (phase_edges[:-1] + phase_edges[1:]) / 2,
        "distance_edges": distance_edges,
        "distance_centers": (distance_edges[:-1] + distance_edges[1:]) / 2,
        "phase_counts": phase_counts,
        "grid_points": grid_points,
    }

In [ ]:
momentum_binning_results = bin_posterior_phase_gaussian(
    [
        sm[:,momentum.latent_dim:].detach().numpy()
        for sm in momentum_results.smoothed_mean
    ], 
    [
        sc[:,momentum.latent_dim:, momentum.latent_dim:].detach().numpy()
        for sc in momentum_results.smoothed_cov
    ],
    raw_data.environment_size,
    theta_data.ground_truth,
    phase_bin_width=10,
)

In [ ]:
hsepl.plot_across_session_decoding_hist(
    momentum_binning_results['posterior_mass'].T
    / np.sum(momentum_binning_results['posterior_mass'].T, axis=0, keepdims=True)
)